# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a reproducible workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Get the metadata as a Python object
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {metadata['name']}\n")
print(f"Description: {metadata['description']}\n")
print(f"Published: {metadata.get('datePublished')}\n")
print(f"Dataset ID (@id): {metadata['@id']}\n")
print(f"Dataset license: {metadata['license']}\n")

## 2. Data Overview
Review available record sets, their fields, columns, and their `@id`s.

A record set corresponds to a table-like structure with fields/columns. Here, we query the dataset for available record sets and provide their unique identifiers.

In [ ]:
# Show all record sets in the dataset with their @id fields
print("Available Record Sets:")
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the dataset metadata.\nIf this is unexpected, check the Croissant schema or contact the data provider.")
else:
    for rs in dataset.record_sets:
        print(f"- {rs['@id']} (name: {rs.get('name')})")

# For illustration, try to print fields and columns for each record set if present
for rs in dataset.record_sets:
    rs_id = rs['@id']
    print(f"\nRecord Set @id: {rs_id}")
    print("Fields:")
    fields = rs.get('field', [])
    for field in (fields if isinstance(fields, list) else [fields]):
        if isinstance(field, dict):
            # Inline field definition
            print(f"  - {field.get('@id')} (name: {field.get('name')}, type: {field.get('dataType')})")
        elif isinstance(field, str):
            # Only an @id reference. Try to resolve full definition.
            try:
                definition = dataset._find_by_id(field)
                print(f"  - {field} (name: {definition.get('name')}, type: {definition.get('dataType')})")
            except Exception:
                print(f"  - {field} (no further metadata available)")
    print("Columns:")
    columns = rs.get('column', [])
    for col in (columns if isinstance(columns, list) else [columns]):
        if isinstance(col, dict):
            print(f"  - {col.get('@id')} (name: {col.get('name')}, type: {col.get('dataType')})")
        elif isinstance(col, str):
            try:
                definition = dataset._find_by_id(col)
                print(f"  - {col} (name: {definition.get('name')}, type: {definition.get('dataType')})")
            except Exception:
                print(f"  - {col} (no further metadata available)")

if not record_sets:
    print("No record sets detected. Automatic extraction not possible without record set definitions.")

## 3. Data Extraction

Load data from a specific record set (using its `@id`) into a DataFrame for analysis. All referencing is by the entity's `@id`.

If no record sets are detected above, check dataset documentation for direct file references or potential schema issues.

In [ ]:
# Retrieve all record set @ids
record_sets = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

if not record_sets:
    print("No record sets to extract data from.")
else:
    # List to track which record_set loaded
    print("Extracting DataFrames from the following record sets (by @id):")
    for rs_id in record_sets:
        print(f" - {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if len(records) == 0:
            print(f"[WARNING] No records loaded for record set {rs_id}")
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
    # Pick the first available dataframe for further example
    if dataframes:
        main_rs_id = record_sets[0]
        print(f"\nColumns for record set {main_rs_id}:")
        print(dataframes[main_rs_id].columns.tolist())
        display(dataframes[main_rs_id].head())
    else:
        print("No DataFrames created; data might be missing.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering by field value, normalizing a numeric column, and grouping by a categorical attribute. All operations reference columns using their `@id`s as shown previously.

In [ ]:
import numpy as np
# Choose the main record set for EDA

if not record_sets:
    print("No record sets or DataFrames loaded. EDA not possible.")
else:
    record_set_id = main_rs_id
    df = dataframes[record_set_id]
    print(f"Columns for record set {record_set_id}:\n{df.columns.tolist()}")

    # Attempt to pick a numeric field by examining types
    numeric_field_id = None
    for col in df.columns:
        # Attempt to select a numeric column
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found in the record set for EDA.")
    else:
        print(f"Using numeric field '@id': {numeric_field_id}")

        # Filtering (e.g., values above the 10th percentile or a threshold)
        threshold = df[numeric_field_id].quantile(0.90)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Pick a group (categorical) column different from the numeric one
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object and len(df[col].unique()) < 30:
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by field '@id': {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization

Visualize the distribution of a numeric variable and/or relationships with a categorical group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if not record_sets:
    print("No data to visualize.")
elif numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # If group_field exists, show boxplot
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook showed how to load FAIR² rangeland knowledge adoption dataset metadata and records using the Croissant schema and the `mlcroissant` library.
- We referenced all dataset entities (record sets, fields, columns) by their `@id`.
- After loading, we explored columns, selected a numeric field, filtered and normalized data, and grouped/visualized by a categorical variable when available.
- For custom analyses, repeat the EDA and visualization steps with your fields of interest. Always refer to fields and structures using their `@id` as defined in the Croissant schema.